# Choosing Visual Forms — instructor solution

This executable version demonstrates the transformation, checks and three defensible chart candidates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(36104)


In [ ]:
months = pd.date_range('2025-01-01', periods=12, freq='MS')
messy = pd.DataFrame({
    'Month': months.strftime('%b-%Y'),
    'Bus passengers': rng.integers(820, 1280, 12).astype(str),
    'Train passengers': rng.integers(1050, 1650, 12).astype(str),
    'Ferry passengers': rng.integers(180, 430, 12).astype(str),
    'Bus delay min': np.round(rng.normal(7.5, 1.6, 12), 1),
    'Train delay min': np.round(rng.normal(5.5, 1.2, 12), 1),
    'Ferry delay min': np.round(rng.normal(4.0, 1.0, 12), 1),
})
messy.loc[4, 'Train passengers'] = '..'
messy.loc[8, 'Bus passengers'] = '1,204'
messy.loc[10, 'Ferry delay min'] = np.nan
messy


## Prediction

Twelve months and three modes should produce 36 observations. One passenger value and one delay value should remain missing. Missing is not equivalent to zero.

In [ ]:
passenger_columns = ['Bus passengers', 'Train passengers', 'Ferry passengers']
delay_columns = ['Bus delay min', 'Train delay min', 'Ferry delay min']

passengers = messy.melt(
    id_vars='Month', value_vars=passenger_columns,
    var_name='mode', value_name='passengers',
)
passengers['mode'] = passengers['mode'].str.replace(' passengers', '', regex=False)
passengers['passengers'] = pd.to_numeric(
    passengers['passengers'].str.replace(',', '', regex=False).replace('..', np.nan),
    errors='coerce',
)

delays = messy.melt(
    id_vars='Month', value_vars=delay_columns,
    var_name='mode', value_name='average_delay_minutes',
)
delays['mode'] = delays['mode'].str.replace(' delay min', '', regex=False)

tidy = passengers.merge(delays, on=['Month', 'mode'], validate='one_to_one')
tidy['month'] = pd.to_datetime(tidy.pop('Month'), format='%b-%Y')
tidy = tidy[['month', 'mode', 'passengers', 'average_delay_minutes']].sort_values(['month', 'mode']).reset_index(drop=True)
tidy.head()


In [ ]:
assert tidy.shape == (36, 4)
assert list(tidy.columns) == ['month', 'mode', 'passengers', 'average_delay_minutes']
assert set(tidy['mode']) == {'Bus', 'Train', 'Ferry'}
assert pd.api.types.is_datetime64_any_dtype(tidy['month'])
assert pd.api.types.is_numeric_dtype(tidy['passengers'])
assert pd.api.types.is_numeric_dtype(tidy['average_delay_minutes'])
assert tidy['passengers'].isna().sum() == 1
assert tidy['average_delay_minutes'].isna().sum() == 1
print('Tidy-data checks passed.')


## Task classification

- Most passengers overall: magnitude or ranking.
- Pattern through the year: change over time.
- Passengers versus delay: correlation.
- Variability of delay: distribution.

In [ ]:
mode_colours = {'Bus': '#18678f', 'Train': '#e66852', 'Ferry': '#30a8b1'}

annual = tidy.groupby('mode', as_index=False)['passengers'].sum().sort_values('passengers')
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(annual['mode'], annual['passengers'], color=[mode_colours[m] for m in annual['mode']])
ax.set(title='Which mode carried the most passengers?', xlabel='Annual passengers', ylabel='')
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
for mode, group in tidy.groupby('mode'):
    ax.plot(group['month'], group['passengers'], marker='o', label=mode, color=mode_colours[mode])
ax.set(title='How did passenger use change through the year?', xlabel='', ylabel='Monthly passengers')
ax.legend(frameon=False, ncol=3)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
for mode, group in tidy.groupby('mode'):
    ax.scatter(group['passengers'], group['average_delay_minutes'], label=mode, color=mode_colours[mode], s=55)
ax.set(title='Do busier months also have longer delays?', xlabel='Monthly passengers', ylabel='Average delay (minutes)')
ax.legend(frameon=False)
plt.show()


## Suggested decision

For an operations manager asking which mode carried the most passengers, select the ordered horizontal bar chart. Position and length on a common scale make the ranking direct. Reject the line chart for this question because it preserves useful temporal detail but makes the annual comparison slower and less direct.

Remaining limitation: summing the available months does not adjust for the missing train observation.

## Provenance model

A useful record identifies the requested transformation, notes that missing values were preserved, records the structural assertions, and explains why the generated chart was modified or rejected.